# previews

> Keeping preview frames out of git and regenerating them for the docs

In [ ]:
#| default_exp previews

`demo` stores a preview in a notebook as two representations: the frame as HTML, and the markup as Markdown. The frame holds the whole stylesheet and script, so a notebook with many previews is large, and every style change rewrites every frame. The tools here keep frames out of git while leaving them in the working copy, and regenerate them when the docs are built.

In [ ]:
import json, sys, re, tempfile
from fastcore.utils import *
from fastcore.script import call_parse
from fastcore.nbio import read_nb, dict2nb, nb2str, join_out
from nbdev.process import Processor

In [ ]:
from fastcore.test import *
from fastcore.nbio import mk_cell, new_nb, write_nb
from answerui.demo import demo
from fasthtml.common import P

## Finding frames

A frame is an output whose HTML root carries `answerui-preview`. The attribute is in the content, so it survives every notebook cleaner:

In [ ]:
_frame_re = re.compile(r'^\s*<div[^>]*\banswerui-preview\b')

def is_preview(out):
    "Does output `out` hold a preview frame?"
    return bool(_frame_re.match(join_out(out.get('data', {}).get('text/html', ''))))

`strip_previews` removes the frame from every preview output in a notebook and keeps the Markdown, so the file still records what each preview showed:

In [ ]:
def strip_previews(nb):
    "Remove the frame from every preview output in `nb`, keeping the Markdown markup; returns the count"
    n = 0
    for c in nb.cells:
        for o in c.get('outputs', []):
            if is_preview(o):
                del o['data']['text/html']
                n += 1
    return n

In [ ]:
p = demo(P('Hello'))
cell = mk_cell("demo(P('Hello'))", outputs=[dict(output_type='execute_result', data=p._repr_mimebundle_(), metadata={}, execution_count=1)])
nb = new_nb([cell])
test_eq(strip_previews(nb), 1)
test_eq(list(nb.cells[0].outputs[0].data), ['text/markdown'])
test_eq(strip_previews(nb), 0)

## The git filter

`answerui-clean` is a git clean filter: a notebook in on stdin, the same notebook out on stdout without its frames. Git runs it when it stages a notebook, so the working copy keeps every frame and the repository stores none. A checkout brings notebooks back without frames, and the previews return when the cells run.

In [ ]:
@call_parse
def answerui_clean():
    "Git clean filter: a notebook from stdin to stdout without preview frames"
    nb = dict2nb(json.load(sys.stdin))
    strip_previews(nb)
    sys.stdout.write(nb2str(nb))

`answerui-install-hooks` sets the filter up in the current repository. It maps `*.ipynb` to the filter in `.gitattributes`, which is committed, and defines the filter in the repository's git config, which is not, so run it once in each clone. The filter is `required`, so git refuses to stage a notebook when `answerui-clean` is missing instead of committing frames.

In [ ]:
_filter = 'answerui-preview'
_attr = f'*.ipynb filter={_filter}'

def install_hooks(root):
    "Install the preview clean filter in the git repository at `root`"
    attrs = Path(root)/'.gitattributes'
    txt = attrs.read_text() if attrs.exists() else ''
    if _attr not in txt.splitlines(): attrs.write_text(txt + ('\n' if txt and not txt.endswith('\n') else '') + _attr + '\n')
    run(f'git -C {root} config filter.{_filter}.clean answerui-clean')
    run(f'git -C {root} config filter.{_filter}.smudge cat')
    run(f'git -C {root} config filter.{_filter}.required true')

@call_parse
def answerui_install_hooks():
    "Install the preview clean filter in the current git repository"
    install_hooks(run('git rev-parse --show-toplevel').strip())
    print('Preview filter installed. Run `git add --renormalize .` to restage existing notebooks.')

Installing into a fresh repository twice adds the attribute line once and defines the filter:

In [ ]:
with tempfile.TemporaryDirectory() as d:
    run(f'git -C {d} init -q')
    install_hooks(d)
    install_hooks(d)
    test_eq((Path(d)/'.gitattributes').read_text(), '*.ipynb filter=answerui-preview\n')
    test_eq(run(f'git -C {d} config --get filter.answerui-preview.clean').strip(), 'answerui-clean')
    test_eq(run(f'git -C {d} config --get filter.answerui-preview.smudge').strip(), 'cat')

## Regenerating previews for the docs

With frames stripped, the docs build has to render them again. nbdev runs the processors named in `doc_procs` on every notebook it builds. `exec_previews` runs `nbdev-test --save` on a copy of the notebook beside it and gives every code cell in the processed notebook the outputs of that run, so no stored output goes stale on the site.

The run is a separate process for each notebook. A docs worker, like a test worker, runs several notebooks in one process, and `answerui.registry` stays imported between them, so an in-process run would carry one notebook's rules into the next notebook's previews. For the same reason, `nbdev-test --save` across a whole project can save previews holding rules from other notebooks in the same worker; run it one notebook at a time when the saved previews matter.

Add the processor in `pyproject.toml`:

```toml
[tool.nbdev]
doc_procs = ["answerui.previews:exec_previews"]
```

The docs build needs nbdev, which the `dev` extra provides.

In [ ]:
def exec_outputs(path):
    "Outputs by cell id from running the notebook at `path` with `nbdev-test --save`, in its own process"
    path = Path(path)
    with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.', suffix='.ipynb', delete=False) as f: tmp = Path(f.name)
    try:
        tmp.write_bytes(path.read_bytes())
        run(f'nbdev-test "{tmp}" --save')
        return {c.id: c.outputs for c in read_nb(tmp).cells if c.cell_type=='code'}
    finally: tmp.unlink()

class exec_previews(Processor):
    "Docs processor: run the notebook and give every code cell the outputs of that run"
    def begin(self): self.outs = exec_outputs(self.nb.path_)
    def cell(self, cell):
        if cell.cell_type=='code' and cell.id in self.outs: cell.outputs = self.outs[cell.id]

A run of a small notebook gives its preview cell a frame:

In [ ]:
nb = new_nb([mk_cell('from answerui.demo import demo\nfrom fasthtml.common import P'), mk_cell("demo(P('Hello'))", id='preview')])
with tempfile.TemporaryDirectory() as d:
    write_nb(nb, Path(d)/'small.ipynb')
    outs = exec_outputs(Path(d)/'small.ipynb')
test_eq(list(outs['preview'][0]['data']), ['text/html', 'text/markdown', 'text/plain'])

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()